## RAG over KG triples

Here we use **verbalized entity-centric chunks** from a linearized JSONL file that we got from `kg_linearization.ipynb`. 

The flow is:

1. Stream triples and build **entity buckets** for each focal id (e.g `event/...`, `match/...`, `player/...`,...) collects lines where it appears as **subject** or as **object** of another triple. 
2. Uses **verbalization template** made from `ontology.ttl`.
3. Embed with **BGE-M3 via Ollama** into Chroma.

In [1]:
# Install dependencies (run once)
!pip install -q chromadb requests openai rdflib


In [2]:
from pathlib import Path
import json
import os
import re
import hashlib
from typing import Dict, List, Any, Tuple, Optional

import requests
import chromadb
from chromadb.config import Settings
from openai import OpenAI

project_root = Path('.').resolve()
jsonl_path = project_root / 'fcdf_kg_triples_linearized.jsonl'

# ---- LLM (Open WebUI, OpenAI-compatible) ----
LLM_BASE_URL = os.getenv('LLM_BASE_URL', 'https://web.ollama-gpt-oss.ai.wu.ac.at/api')
LLM_API_KEY = os.getenv('LLM_API_KEY', 'sk-9b41b856cc0b403b8a3c10618f1c2996')
LLM_MODEL = 'gpt-oss:120b'

llm_client = OpenAI(
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL,
    timeout=120.0,
    max_retries=4,
)

# ---- Embeddings (Ollama, BGE-M3) ----
OLLAMA_EMBED_BASE = os.getenv('OLLAMA_EMBED_BASE', 'http://localhost:11434')
EMBED_MODEL = os.getenv('EMBED_MODEL', 'bge-m3')

print('Project root:', project_root)
print('JSONL exists:', jsonl_path.exists(), jsonl_path)
print('LLM base URL:', LLM_BASE_URL)
print('LLM model:', LLM_MODEL)
print('Ollama embed base:', OLLAMA_EMBED_BASE)
print('Embed model:', EMBED_MODEL)

# ---- Ontology + entity-centric indexing ----
ontology_path = project_root / 'ontology.ttl'
COLLECTION_NAME = 'kg_entities_ontology_sliced'

print('Ontology exists:', ontology_path.exists(), ontology_path)
print('Chroma collection:', COLLECTION_NAME)


Project root: C:\Users\dyury\Desktop\Master Thesis
JSONL exists: True C:\Users\dyury\Desktop\Master Thesis\fcdf_kg_triples_linearized.jsonl
LLM base URL: https://web.ollama-gpt-oss.ai.wu.ac.at/api
LLM model: gpt-oss:120b
Ollama embed base: http://localhost:11434
Embed model: bge-m3
Ontology exists: True C:\Users\dyury\Desktop\Master Thesis\ontology.ttl
Chroma collection: kg_entities_ontology_sliced


In [ ]:
# class/property map from ontology.ttl, subject+object buckets,
# deterministic verbalization.
from __future__ import annotations
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any, Callable, DefaultDict, Iterable, Iterator, Sequence, Set

from rdflib import Graph, URIRef
from rdflib.namespace import OWL, RDF, RDFS

CORE_NS = 'https://w3id.org/football-cdf/core#'

# StatsBomb-style `type` literals -> OWL class local names (ontology.ttl).
DEFAULT_TYPE_LITERAL_TO_CLASS: Dict[str, str] = {
    'event': 'Event',
    'match': 'Match',
    'pass': 'Pass',
    'shot': 'Shot',
    'goal': 'Goal',
    'card': 'Card',
    'whistle': 'Whistle',
    'carry': 'Event',
    'pressure': 'Event',
    'ball receipt*': 'Event',
    'ball receipt': 'Event',
    'ball recovery': 'Event',
    'duel': 'Event',
    'block': 'Event',
    'dribble': 'Event',
    'foul committed': 'Event',
    'foul won': 'Event',
    'clearance': 'Event',
    'interception': 'Event',
    'miscontrol': 'Event',
    'dispossessed': 'Event',
    'dribbled past': 'Event',
    'goal keeper': 'Event',
    'injury stoppage': 'Event',
    'shield': 'Event',
    '50/50': 'Event',
    'tactical shift': 'Event',
    'referee ball-drop': 'Event',
    'player on': 'Event',
    'player off': 'Event',
    'half start': 'Event',
    'half end': 'Event',
    'bad behaviour': 'Event',
    'miscellaneous event': 'Misc',
    'subtitution': 'Subtitution',
    'substitution': 'Subtitution',
    'match result': 'Match_Result',
    'player': 'Player',
    'team': 'Team',
    'referee': 'Referee',
}

DEFAULT_RESOURCE_PREFIXES: Tuple[str, ...] = (
    'event/', 'match/', 'player/', 'team/', 'referee/', 'season/',
    'match_result/', 'period/', 'whistle/', 'meta/', 'competition/', 'vendor/',
)

# match/* rows whose predicate lists linked events -> merged into events slices.
# _MATCH_EVENTISH_PREDS = frozenset({'events', 'event_goals', 'event_cards', 'event_substitutions'})

# match/* metadata predicates -> single match_core slice.
_MATCH_CORE_PREDS = frozenset({
    'id', 'type', 'Kickoff', 'kickoff_time', 'competition', 'season', 'referee',
    'team_home', 'team_away', 'match_result', 'match_status', 'stadium_id', 'stadium',
    'video_metadata', 'vendor', 'meta_landmarks', 'meta_meta',
})


def parse_triple_text(line_text: str) -> Tuple[str, str, str]:
    parts = line_text.strip().split(' ')
    if len(parts) < 3:
        return (parts[0] if parts else 'unknown', 'unknown', '')
    return parts[0], parts[1], ' '.join(parts[2:])


def _short_uri(uri: str) -> str:
    if uri.startswith(CORE_NS):
        return 'core:' + uri.rsplit('#', 1)[-1]
    return uri


def _local_name(uri: str) -> str:
    if '#' in uri:
        return uri.rsplit('#', 1)[-1]
    return uri.rsplit('/', 1)[-1]


def _expand_domain_or_range(g: Graph, node) -> List[str]:
    out: List[str] = []
    if isinstance(node, URIRef):
        out.append(_short_uri(str(node)))
        return out
    for u in g.objects(node, OWL.unionOf):
        cur = u
        while cur is not None and cur != RDF.nil:
            for first in g.objects(cur, RDF.first):
                if isinstance(first, URIRef):
                    out.append(_short_uri(str(first)))
            rests = list(g.objects(cur, RDF.rest))
            cur = rests[0] if rests else None
    return out


@dataclass
# Per core:Class — property local names in ontology declaration order.
class ClassPropertyIndex:
    class_to_props: Dict[str, List[str]] = field(default_factory=dict)
    prop_labels: Dict[str, str] = field(default_factory=dict)
    class_labels: Dict[str, str] = field(default_factory=dict)

    def props_for_class(self, class_local: str) -> List[str]:
        return list(self.class_to_props.get(class_local, ()))


def build_class_property_index(ontology_path: Path) -> ClassPropertyIndex:
    g = Graph()
    g.parse(str(ontology_path), format='turtle')
    idx = ClassPropertyIndex()
    for s in g.subjects(RDF.type, OWL.Class):
        if not isinstance(s, URIRef) or not str(s).startswith(CORE_NS):
            continue
        ln = _local_name(str(s))
        lab = next((str(x) for x in g.objects(s, RDFS.label)), '')
        idx.class_labels[ln] = lab or ln
        idx.class_to_props.setdefault(ln, [])

    decl_order: List[Tuple[str, str, URIRef]] = []
    for p in g.subjects(RDF.type, OWL.ObjectProperty):
        if isinstance(p, URIRef) and str(p).startswith(CORE_NS):
            decl_order.append(('obj', _local_name(str(p)), p))
    for p in g.subjects(RDF.type, OWL.DatatypeProperty):
        if isinstance(p, URIRef) and str(p).startswith(CORE_NS):
            decl_order.append(('dt', _local_name(str(p)), p))

    seen: Set[Tuple[str, str]] = set()
    for _kind, prop_ln, p_uri in decl_order:
        lab = next((str(x) for x in g.objects(p_uri, RDFS.label)), prop_ln)
        idx.prop_labels[prop_ln] = lab
        doms: List[str] = []
        for d in g.objects(p_uri, RDFS.domain):
            doms.extend(_expand_domain_or_range(g, d))
        for cls in sorted(set(doms)):
            if not cls.startswith('core:'):
                continue
            cl = cls.split(':', 1)[1]
            if (cl, prop_ln) in seen:
                continue
            seen.add((cl, prop_ln))
            idx.class_to_props.setdefault(cl, []).append(prop_ln)
    return idx


def is_resource_object(obj: str, prefixes: Sequence[str] = DEFAULT_RESOURCE_PREFIXES) -> bool:
    return any(obj.strip().startswith(p) for p in prefixes)


def is_focal_candidate(subj: str, prefixes: Sequence[str] = DEFAULT_RESOURCE_PREFIXES) -> bool:
    return any(subj.strip().startswith(p) for p in prefixes)


def resolve_owl_class_for_focal(focal: str, type_literals: Sequence[str], type_map: Dict[str, str]) -> str:
    PREFIX_TO_CLASS = {
        'match/': 'Match', 'player/': 'Player', 'team/': 'Team', 'referee/': 'Referee',
        'season/': 'Season', 'competition/': 'Competition', 'match_result/': 'Match_Result',
        'whistle/': 'Whistle', 'meta/': 'Meta', 'vendor/': 'Vendor',
    }
    for pref, cls in PREFIX_TO_CLASS.items():
        if focal.startswith(pref):
            return cls
    if focal.startswith('event/'):
        mapped = []
        for t in type_literals:
            key = t.strip().lower()
            if key in type_map:
                mapped.append(type_map[key])
        for cls in mapped:
            if cls != 'Event':
                return cls
        return mapped[0] if mapped else 'Event'
    return 'Event'


def _facts_from_lines(lines: Iterable[str], focal: str) -> Tuple[Dict[str, List[str]], List[str], List[str]]:
    primary: DefaultDict[str, List[str]] = defaultdict(list)
    incoming: List[str] = []
    type_literals: List[str] = []
    for line in lines:
        s, p, o = parse_triple_text(line)
        if s == focal:
            if p == 'type':
                type_literals.append(o)
            primary[p].append(o)
        else:
            incoming.append(line)
    return dict(primary), incoming, type_literals


# ---- Prose verbalization helpers ----

def _get(facts: Dict[str, List[str]], prop: str, default: str = '') -> str:
    vals = facts.get(prop)
    return vals[0] if vals else default


def _getall(facts: Dict[str, List[str]], prop: str) -> List[str]:
    return facts.get(prop, [])


def _entity_display_name(facts: Dict[str, List[str]]) -> str:
    return _get(facts, 'label') or _get(facts, 'name')


def _fmt_coords(facts: Dict[str, List[str]]) -> str:
    x, y = _get(facts, 'x'), _get(facts, 'y')
    if not x and not y:
        return ''
    xe, ye = _get(facts, 'x_end'), _get(facts, 'y_end')
    start = f'({x}, {y})' if x and y else f'({x or y})'
    if xe and ye:
        return f'from {start} toward ({xe}, {ye})'
    return f'at pitch position {start}'


def _fmt_incoming_refs(incoming: List[str], focal: str) -> str:
    if not incoming:
        return ''
    parts = []
    for line in incoming:
        s, p, _ = parse_triple_text(line)
        parts.append(f'{s} [{p}]')
    return 'Referenced by: ' + '; '.join(parts) + '.'


def _base_event_ctx(focal: str, facts: Dict[str, List[str]], incoming: List[str]):
    player = _get(facts, 'player_id')
    team = _get(facts, 'team_id')
    period = _get(facts, 'event_period')
    time_ = _get(facts, 'time')
    match_ref = next(
        (parse_triple_text(l)[0] for l in incoming
         if parse_triple_text(l)[0].startswith('match/')), '')
    who = ''
    if player:
        who = f'by player {player}'
        if team:
            who += f' ({team})'
    ctx_parts = []
    if period:
        ctx_parts.append(f'during the {period}')
    if time_:
        ctx_parts.append(f'at {time_}')
    if match_ref:
        ctx_parts.append(f'in match {match_ref}')
    return who, ', '.join(ctx_parts), match_ref


def _render_base_event_lines(focal: str, facts: Dict[str, List[str]],
                              incoming: List[str], prefix: str) -> List[str]:
    who, ctx, _ = _base_event_ctx(focal, facts, incoming)
    line = f'Event {focal} is a {prefix}'
    if who:
        line += f', performed {who}'
    if ctx:
        line += f', {ctx}'
    line += '.'
    parts = [line]
    coords = _fmt_coords(facts)
    if coords:
        parts.append(f'Position: {coords}.')
    body = _get(facts, 'body_part')
    if body:
        parts.append(f'Body part: {body}.')
    related = _getall(facts, 'related_event_ids')
    if related:
        parts.append(f'Related events: {", ".join(related)}.')
    return parts


def _render_pass(focal, facts, incoming, cls_index, type_map) -> str:
    pass_type = _get(facts, 'pass_type')
    label = f'{pass_type.replace("_", " ")} pass' if pass_type and pass_type != 'None' else 'pass'
    parts = _render_base_event_lines(focal, facts, incoming, label)
    receiver = _get(facts, 'receiver_id')
    recv_time = _get(facts, 'receiver_time')
    outcome = _get(facts, 'pass_outcome_type') or _get(facts, 'outcome_type')
    if receiver:
        s = f'Received by {receiver}'
        if recv_time:
            s += f' at {recv_time}'
        parts.append(s + '.')
    if outcome:
        parts.append(f'Outcome: {outcome}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_shot(focal, facts, incoming, cls_index, type_map) -> str:
    shot_type = _get(facts, 'shot_type')
    label = f'{shot_type.replace("_", " ")} shot' if shot_type and shot_type != 'None' else 'shot'
    parts = _render_base_event_lines(focal, facts, incoming, label)
    outcome = _get(facts, 'shot_outcome_type') or _get(facts, 'outcome_type')
    if outcome:
        parts.append(f'Outcome: {outcome}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_goal(focal, facts, incoming, cls_index, type_map) -> str:
    parts = _render_base_event_lines(focal, facts, incoming, 'goal')
    assist = _get(facts, 'assist_id')
    is_own = _get(facts, 'is_own_goal')
    is_pen = _get(facts, 'is_penalty')
    outcome = _get(facts, 'shot_outcome_type') or _get(facts, 'outcome_type')
    if assist:
        parts.append(f'Assisted by {assist}.')
    if is_own and is_own.lower() == 'true':
        parts.append('Own goal.')
    if is_pen and is_pen.lower() == 'true':
        parts.append('Penalty kick.')
    if outcome:
        parts.append(f'Shot outcome: {outcome}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_misc(focal, facts, incoming, cls_index, type_map) -> str:
    misc_type = _get(facts, 'misc_type') or _get(facts, 'sub_type')
    label = misc_type.replace('_', ' ') if misc_type else 'miscellaneous action'
    parts = _render_base_event_lines(focal, facts, incoming, label)
    outcome = _get(facts, 'misc_outcome_type') or _get(facts, 'outcome_type')
    if outcome:
        parts.append(f'Outcome: {outcome}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_whistle(focal, facts, incoming, cls_index, type_map) -> str:
    w_type = _get(facts, 'whistle_type') or _get(facts, 'sub_type')
    label = w_type.replace('_', ' ') if w_type else 'referee action'
    parts = _render_base_event_lines(focal, facts, incoming, label)
    outcome = _get(facts, 'whistle_outcome_type') or _get(facts, 'outcome_type')
    if outcome:
        parts.append(f'Outcome: {outcome}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_card(focal, facts, incoming, cls_index, type_map) -> str:
    card_type = _get(facts, 'card_type')
    label = f'{card_type.replace("_", " ")} card' if card_type else 'card'
    parts = _render_base_event_lines(focal, facts, incoming, label)
    w_type = _get(facts, 'whistle_type')
    if w_type:
        parts.append(f'Referee sub-type: {w_type}.')
    outcome = _get(facts, 'whistle_outcome_type')
    if outcome:
        parts.append(f'Outcome: {outcome}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_substitution(focal, facts, incoming, cls_index, type_map) -> str:
    player = _get(facts, 'player_id')
    out_player = _get(facts, 'out_player_id')
    team = _get(facts, 'team_id')
    period = _get(facts, 'event_period')
    time_ = _get(facts, 'time')
    out_time = _get(facts, 'out_time')
    match_ref = next(
        (parse_triple_text(l)[0] for l in incoming
         if parse_triple_text(l)[0].startswith('match/')), '')
    line = f'Event {focal} is a substitution'
    if time_:
        line += f' at {time_}'
    if period:
        line += f' during the {period}'
    if match_ref:
        line += f' in match {match_ref}'
    line += '.'
    parts = [line]
    if player:
        s = f'Player coming on: {player}'
        if team:
            s += f' ({team})'
        parts.append(s + '.')
    if out_player:
        parts.append(f'Player going off: {out_player}.')
    if out_time:
        parts.append(f'Substitution recorded at {out_time}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_event(focal, facts, incoming, cls_index, type_map) -> str:
    ev_type = _get(facts, 'type')
    sub_type = _get(facts, 'sub_type')
    label = ev_type.replace('_', ' ') if ev_type else 'event'
    if sub_type and sub_type != 'None':
        label += f' ({sub_type.replace("_", " ")})'
    parts = _render_base_event_lines(focal, facts, incoming, label)
    outcome = _get(facts, 'outcome_type')
    if outcome:
        parts.append(f'Outcome: {outcome}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_match(focal, facts, incoming, cls_index, type_map,
                  slice_key: Optional[str] = None) -> str:
    kickoff = _get(facts, 'kickoff_time')
    home = _get(facts, 'team_home')
    away = _get(facts, 'team_away')
    competition = _get(facts, 'competition')
    season = _get(facts, 'season')
    referee = _get(facts, 'referee')
    result = _get(facts, 'match_result')
    status = _get(facts, 'match_status')
    meta = _get(facts, 'meta_video')

    line = f'Match {focal}'
    if kickoff:
        line += f' took place on {kickoff}'
    if home and away:
        line += f', between home side {home} and away side {away}'
    elif home:
        line += f', home team: {home}'
    elif away:
        line += f', away team: {away}'
    if competition:
        line += f', in competition {competition}'
    if season:
        line += f' (season {season})'
    if referee:
        line += f', refereed by {referee}'
    line += '.'

    parts = [line]
    if result:
        parts.append(f'Result: {result}.')
    if status:
        parts.append(f'Match status: {status}.')
    if meta:
        parts.append(f'Video metadata: {meta}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_match_result(focal, facts, incoming, cls_index, type_map) -> str:
    home = _get(facts, 'result_home')
    away = _get(facts, 'result_away')
    period = _get(facts, 'result_period')
    match_ref = next(
        (parse_triple_text(l)[0] for l in incoming
         if parse_triple_text(l)[0].startswith('match/')), '')
    parts = [f'Match result {focal}:']
    if home and away:
        parts.append(f'{home}-{away} (home-away).')
    elif home:
        parts.append(f'Home goals: {home}.')
    elif away:
        parts.append(f'Away goals: {away}.')
    if period:
        parts.append(f'Period: {period}.')
    if match_ref:
        parts.append(f'For match: {match_ref}.')
    return ' '.join(parts)


#def _render_match_status(focal, facts, incoming, cls_index, type_map) -> str:
#    extra = _get(facts, 'has_extratime')
#    shootout = _get(facts, 'has_shootout')
#    neutral = _get(facts, 'is_neutral')
#    match_ref = next(
#        (parse_triple_text(l)[0] for l in incoming
#         if parse_triple_text(l)[0].startswith('match/')), '')
#    parts = [f'Match status {focal}:']
#    details = []
#    if extra:
#        details.append(f'extra time: {extra}')
#    if shootout:
#        details.append(f'shootout: {shootout}')
#    if neutral:
#        details.append(f'neutral venue: {neutral}')
#    if details:
#        parts.append(', '.join(details) + '.')
#    if match_ref:
#        parts.append(f'For match: {match_ref}.')
#    return ' '.join(parts)


def _render_player(focal, facts, incoming, cls_index, type_map) -> str:
    display = _entity_display_name(facts)
    jersey = _get(facts, 'jersey_number')
    starter = _get(facts, 'is_starter')
    played = _get(facts, 'has_played')
    team = _get(facts, 'team_id')
    team_refs = list({parse_triple_text(l)[0] for l in incoming
                      if parse_triple_text(l)[0].startswith('team/')})
    line = f'Player {focal}'
    if display:
        line += f' ({display})'
    if jersey:
        line += f', jersey number {jersey}'
    line += '.'
    parts = [line]
    if starter:
        parts.append(f'Starting lineup: {starter}.')
    if played:
        parts.append(f'Has played: {played}.')
    if team:
        parts.append(f'Team: {team}.')
    elif team_refs:
        parts.append(f'Team id(s): {", ".join(team_refs)}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_team(focal, facts, incoming, cls_index, type_map) -> str:
    display = _entity_display_name(facts)
    players = _getall(facts, 'players')
    line = f'Team {focal}'
    if display:
        line += f' ({display})'
    line += '.'
    parts = [line]
    if players:
        preview = ', '.join(players[:15])
        suffix = ' ...' if len(players) > 15 else ''
        parts.append(f'Squad ({len(players)} players): {preview}{suffix}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_referee(focal, facts, incoming, cls_index, type_map) -> str:
    display = _entity_display_name(facts)
    match_refs = list({parse_triple_text(l)[0] for l in incoming
                       if parse_triple_text(l)[0].startswith('match/')})
    parts = [f'Referee {focal}' + (f' ({display})' if display else '') + '.']
    if match_refs:
        preview = ', '.join(sorted(match_refs)[:10])
        suffix = ' ...' if len(match_refs) > 10 else ''
        parts.append(f'Officiated matches ({len(match_refs)}): {preview}{suffix}.')
    ref = _fmt_incoming_refs(incoming, focal)
    if ref:
        parts.append(ref)
    return ' '.join(parts)


def _render_competition(focal, facts, incoming, cls_index, type_map) -> str:
    name = _get(facts, 'name')
    match_refs = list({parse_triple_text(l)[0] for l in incoming
                       if parse_triple_text(l)[0].startswith('match/')})
    parts = [f'Competition {focal}' + (f': {name}' if name else '') + '.']
    if match_refs:
        parts.append(f'Matches: {", ".join(match_refs)}.')
    return ' '.join(parts)


def _render_season(focal, facts, incoming, cls_index, type_map) -> str:
    name = _get(facts, 'name')
    match_refs = list({parse_triple_text(l)[0] for l in incoming
                       if parse_triple_text(l)[0].startswith('match/')})
    parts = [f'Season {focal}' + (f': {name}' if name else '') + '.']
    if match_refs:
        parts.append(f'Matches in this season: {", ".join(match_refs)}.')
    return ' '.join(parts)


def _render_meta(focal, facts, incoming, cls_index, type_map) -> str:
    vendor = _get(facts, 'vendor')
    version = _get(facts, 'version')
    fps = _get(facts, 'fps')
    collected = _get(facts, 'collection_timing')
    match_ref = next(
        (parse_triple_text(l)[0] for l in incoming
         if parse_triple_text(l)[0].startswith('match/')), '')
    parts = [f'Video metadata {focal}:']
    details = []
    if vendor:
        details.append(f'vendor {vendor}')
    if version:
        details.append(f'version {version}')
    if fps:
        details.append(f'{fps} fps')
    if collected:
        details.append(f'collected at {collected}')
    if details:
        parts.append(', '.join(details) + '.')
    if match_ref:
        parts.append(f'For match: {match_ref}.')
    return ' '.join(parts)


PROSE_TEMPLATE_REGISTRY: Dict[str, Any] = {
    'Pass': _render_pass,
    'Shot': _render_shot,
    'Goal': _render_goal,
    'Misc': _render_misc,
    'Whistle': _render_whistle,
    'Card': _render_card,
    'Subtitution': _render_substitution,
    'Event': _render_event,
    'Match': _render_match,
    'Match_Result': _render_match_result,
    #'Match_Status': _render_match_status,
    'Player': _render_player,
    'Team': _render_team,
    'Referee': _render_referee,
    'Competition': _render_competition,
    'Season': _render_season,
    'Meta': _render_meta,
}


def verbalize_bucket(focal: str, lines: List[str], cls_index: ClassPropertyIndex,
                     type_map: Dict[str, str], *, slice_key: Optional[str] = None) -> str:
    facts, incoming, type_literals = _facts_from_lines(lines, focal)
    owl_class = resolve_owl_class_for_focal(focal, type_literals, type_map)
    renderer = PROSE_TEMPLATE_REGISTRY.get(owl_class)
    if renderer is None:
        # Generic fallback for unknown classes
        class_label = cls_index.class_labels.get(owl_class, owl_class)
        head_order = cls_index.props_for_class(owl_class)
        used: Set[str] = set()
        head_bits: List[str] = []
        for pred in head_order:
            if pred not in facts:
                continue
            used.add(pred)
            vals = facts[pred]
            label = cls_index.prop_labels.get(pred, pred)
            joined = ', '.join(vals) if len(vals) > 1 else vals[0]
            head_bits.append(f'{label}={joined}')
        intro = f"focal={focal} (ontology class core:{owl_class}, rdfs:label='{class_label}')."
        body_parts: List[str] = [intro]
        if head_bits:
            body_parts.append('Properties: ' + '; '.join(head_bits) + '.')
        tail_preds = sorted(p for p in facts if p not in used)
        tail_parts: List[str] = []
        for pred in tail_preds:
            vals = facts[pred]
            joined = ', '.join(vals) if len(vals) > 1 else vals[0]
            tail_parts.append(f'{pred}={joined}')
        if tail_parts:
            body_parts.append('Other properties: ' + '; '.join(tail_parts) + '.')
        inc_parts: List[str] = []
        for line in incoming:
            s, p, _ = parse_triple_text(line)
            inc_parts.append(f'{s} {p} {focal}')
        if inc_parts:
            body_parts.append('Incoming references: ' + '; '.join(inc_parts) + '.')
        return ' '.join(body_parts).strip()
    if owl_class == 'Match':
        return renderer(focal, facts, incoming, cls_index, type_map, slice_key=slice_key)
    return renderer(focal, facts, incoming, cls_index, type_map)


def _match_predicate_family(pred: str) -> str:
   #if pred in _MATCH_EVENTISH_PREDS:
   #     return 'events'
    if pred in _MATCH_CORE_PREDS or pred.startswith('meta_'):
        return 'core'
    if 'period' in pred or 'whistle' in pred or pred.startswith('match/'):
        return 'structure'
    if pred == 'players':
        return 'players'
    return 'other'


# Partition a match/*/ bucket into predicate-family slices; paginate `events`.
def slice_match_bucket(focal: str, lines: List[str], *, max_event_lines: int = 80,
                       max_event_chars: int = 8000) -> List[Tuple[str, List[str]]]:
    assert focal.startswith('match/'), 'slice_match_bucket expects match focal'
    groups: DefaultDict[str, List[str]] = defaultdict(list)
    for line in lines:
        s, p, _ = parse_triple_text(line)
        fam = _match_predicate_family(p)
        if s != focal:
            groups['incoming'].append(line)
        else:
            groups[fam].append(line)

    out: List[Tuple[str, List[str]]] = []
    if groups['core']: out.append(('match_core', groups['core']))
    if groups['structure']: out.append(('match_structure', groups['structure']))
    if groups['players']: out.append(('match_players', groups['players']))

    

    if groups['other']: out.append(('match_other', groups['other']))
    if groups['incoming']: out.append(('incoming_refs', groups['incoming']))
    if not out: out.append(('match_core', []))
    return out


def slice_non_match_bucket(focal: str, lines: List[str]) -> List[Tuple[str, List[str]]]:
    return [('default', lines)]


def stable_doc_base_id(focal: str, slice_key: str) -> str:
    return focal if slice_key == 'default' else f'{focal}::slice::{slice_key}'


def split_verbalized_into_chunks(base_id: str, text: str, max_chars: int = 3500) -> List[Dict[str, Any]]:
    text = (text or '').strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [{'id': f'{base_id}::part::0', 'text': text}]

    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks: List[str] = []
    current = ''
    for sentence in sentences:
        if not sentence:
            continue
        if len(sentence) > max_chars:
            if current:
                chunks.append(current.strip())
                current = ''
            for start in range(0, len(sentence), max_chars):
                chunks.append(sentence[start:start + max_chars])
            continue
        candidate = (current + ' ' + sentence).strip() if current else sentence
        if len(candidate) > max_chars:
            chunks.append(current.strip())
            current = sentence
        else:
            current = candidate
    if current.strip():
        chunks.append(current.strip())
    return [{'id': f'{base_id}::part::{i}', 'text': c} for i, c in enumerate(chunks)]


# --- Bucket streaming ---

def stream_jsonl_triple_texts(path: Path, *, max_lines: Optional[int] = None) -> Iterator[str]:
    n = 0
    with open(path, 'r', encoding='utf-8') as f:
        for raw in f:
            if max_lines is not None and n >= max_lines:
                break
            raw = raw.strip()
            if not raw:
                continue
            try:
                obj = json.loads(raw)
            except json.JSONDecodeError:
                continue
            text = obj.get('text')
            if isinstance(text, str) and text:
                yield text
                n += 1


def stream_entity_buckets_in_memory(path: Path, *, max_lines: Optional[int] = None,
                                    max_focals: Optional[int] = None,
                                    object_pred_allowlist: Optional[Set[str]] = None) -> Dict[str, List[str]]:
    """Single pass over JSONL: subject + (optional) object promotion, deduped per focal."""
    raw: Dict[str, Tuple[List[str], Set[str]]] = {}

    def add(focal: str, ln: str) -> None:
        if focal not in raw:
            if max_focals is not None and len(raw) >= max_focals:
                return
            raw[focal] = ([], set())
        lst, seen = raw[focal]
        if ln not in seen:
            seen.add(ln)
            lst.append(ln)

    for text in stream_jsonl_triple_texts(path, max_lines=max_lines):
        s, p, o = parse_triple_text(text)
        if is_focal_candidate(s):
            add(s, text)
        if is_resource_object(o) and (object_pred_allowlist is None or p in object_pred_allowlist):
            add(o.strip(), text)
    return {k: v[0] for k, v in raw.items()}


def default_verbalize(focal: str, lines: List[str], cls_index: ClassPropertyIndex,
                      type_map: Dict[str, str], slice_key: str) -> str:
    sk = None if slice_key == 'default' else slice_key
    return verbalize_bucket(focal, lines, cls_index, type_map, slice_key=sk)

# Chroma-ready doc dicts with keys: id, text, focal, slice, owl_class, n_lines.
def build_documents_for_focals(buckets: Dict[str, List[str]], cls_index: ClassPropertyIndex,
                                type_map: Dict[str, str], *,
                                verbalize_fn: Callable[..., str] = default_verbalize,
                                max_chars_per_doc: int = 3500) -> List[Dict[str, Any]]:
    docs: List[Dict[str, Any]] = []
    for focal, lines in buckets.items():
        slices = slice_match_bucket(focal, lines) if focal.startswith('match/') else slice_non_match_bucket(focal, lines)
        for slice_key, slines in slices:
            if not slines:
                continue
            base_id = stable_doc_base_id(focal, slice_key)
            _, _, type_literals = _facts_from_lines(slines, focal)
            owl_class = resolve_owl_class_for_focal(focal, type_literals, type_map)
            text = verbalize_fn(focal, slines, cls_index, type_map, slice_key)
            for chunk in split_verbalized_into_chunks(base_id, text, max_chars=max_chars_per_doc):
                docs.append({
                    'id': chunk['id'],
                    'text': chunk['text'],
                    'focal': focal,
                    'slice': slice_key,
                    'owl_class': owl_class,
                    'n_lines': len(slines),
                })
    return docs


# Build the ontology index and a small preview
CLASS_INDEX = build_class_property_index(ontology_path)
TYPE_MAP = dict(DEFAULT_TYPE_LITERAL_TO_CLASS)
print('OWL classes indexed:', len(CLASS_INDEX.class_labels),
      '| example Match props:', CLASS_INDEX.props_for_class('Match')[:8])

_preview = stream_entity_buckets_in_memory(jsonl_path, max_lines=3000, max_focals=3)
_k = next(iter(_preview))
print('Preview focal:', _k, '| lines:', len(_preview[_k]))
print('Example line:', _preview[_k][0][:200])


OWL classes indexed: 18 | example Match props: ['competition', 'events', 'events_cards', 'events_goals', 'events_subtitutions', 'match_result', 'match_status', 'meta_video']
Preview focal: event/4279d9d6-b511-4e00-a000-02a68505909c | lines: 1
Example line: event/4279d9d6-b511-4e00-a000-02a68505909c x 76.3


In [4]:
def call_llm(prompt: str, model: str = LLM_MODEL) -> str:
    resp = llm_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
    )
    return (resp.choices[0].message.content or '').strip()


# ---- Indexing knobs ----
# Set MAX_LINES to a small number (e.g. 50_000) for a quick smoke run.
MAX_LINES: Optional[int] = None   # None = read every JSONL line
MAX_FOCALS: Optional[int] = None  # None = keep every distinct focal
MAX_CHARS_PER_DOC = 3500

buckets = stream_entity_buckets_in_memory(
    jsonl_path,
    max_lines=MAX_LINES,
    max_focals=MAX_FOCALS,
    object_pred_allowlist=None,
)
print(f'Focals in bucket store: {len(buckets)}')
docs = build_documents_for_focals(
    buckets,
    CLASS_INDEX,
    TYPE_MAP,
    max_chars_per_doc=MAX_CHARS_PER_DOC,
)

print('Docs:', len(docs))
if docs:
    print('Example doc id:', docs[0]['id'])
    print('Example metadata keys:', list(docs[0].keys()))
    print('Example text (400 chars):\n', docs[0]['text'][:400])
else:
    print('No docs were produced.')


Focals in bucket store: 138464
Docs: 138464
Example doc id: event/4279d9d6-b511-4e00-a000-02a68505909c::part::0
Example metadata keys: ['id', 'text', 'focal', 'slice', 'owl_class', 'n_lines']
Example text (400 chars):
 Event event/4279d9d6-b511-4e00-a000-02a68505909c is a pass, performed by player Alejandro Grimaldo García (Bayer Leverkusen), during the first_half, at 00:26:34.526, in match match/3895292. Position: from (76.3, 1.7) toward (87.1, 10.3). Body part: left_foot. Related events: event/34cc58ee-a2de-4f85-b794-aba34846be9e. Received by Florian Wirtz at 00:26:35.301. Referenced by: event/34cc58ee-a2de-4f


In [5]:
import json
from pathlib import Path

out_path = Path("docs_export.json")  # or any path you want

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)

In [6]:
class OllamaEmbeddingFunction:
    # Minimal embedding function wrapper for Chroma using Ollama.
    def __init__(self, model: str = EMBED_MODEL, base_url: str = OLLAMA_EMBED_BASE):
        self.model = model
        self.url = f'{base_url}/api/embeddings'
    def _embed_texts(self, texts: List[str]) -> List[List[float]]:
        vectors: List[List[float]] = []
        for text in texts:
            resp = requests.post(self.url, json={'model': self.model, 'prompt': text}, timeout=120)
            resp.raise_for_status()
            vectors.append(resp.json()['embedding'])
        return vectors
    def embed_documents(self, input: List[str]) -> List[List[float]]:
        return self._embed_texts(input)
    def embed_query(self, input):
        if isinstance(input, str):
            return self._embed_texts([input])[0]
        return self._embed_texts(input)
    def __call__(self, input: List[str]) -> List[List[float]]:
        return self.embed_documents(input)
    def name(self) -> str:
        # Chroma uses this to namespace the collection; include a hash of the
        # model name so switching embed models forces a clean re-index path.
        h = hashlib.sha1(self.model.encode('utf-8')).hexdigest()[:8]
        return f'ollama-{self.model.replace(":", "-")}-{h}'

In [ ]:
# Create Chroma collection
chroma_client = chromadb.PersistentClient(path=str(project_root / 'chroma_db'))
embedding_function = OllamaEmbeddingFunction(model=EMBED_MODEL, base_url=OLLAMA_EMBED_BASE)

RESET_COLLECTION = True  # True wipes the collection before indexing
SKIP_IF_COMPLETE = False   # skip Ollama embeds when Chroma already has all doc ids
BATCH_SIZE = 500

if RESET_COLLECTION:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function,
)

ids_all = [d['id'] for d in docs]
docs_all = [d['text'] for d in docs]
metas_all = [
    {
        'focal': d['focal'],
        'slice': d['slice'],
        'owl_class': d['owl_class'],
        'n_lines': int(d['n_lines']),
    }
    for d in docs
]

existing_count = collection.count()
print(f'Chroma count before indexing: {existing_count} / {len(docs_all)}')

if SKIP_IF_COMPLETE and existing_count >= len(docs_all):
    print('Skipping indexing: collection already has enough vectors (RESET_COLLECTION=False does not skip this loop by itself).')
else:
    # Only embed ids not already stored (add with duplicate ids re-embeds everything).
    existing_ids = set(collection.get(include=[])['ids'])
    pending = [i for i, doc_id in enumerate(ids_all) if doc_id not in existing_ids]
    if not pending:
        print('Skipping indexing: all doc ids already in Chroma.')
    else:
        print(f'Indexing {len(pending)} new docs ({len(existing_ids)} already present)')
        for batch_start in range(0, len(pending), BATCH_SIZE):
            batch_idx = pending[batch_start : batch_start + BATCH_SIZE]
            collection.add(
                ids=[ids_all[i] for i in batch_idx],
                documents=[docs_all[i] for i in batch_idx],
                metadatas=[metas_all[i] for i in batch_idx],
            )
            print(f'Indexed {min(batch_start + BATCH_SIZE, len(pending))}/{len(pending)} new')

print('Chroma count after indexing:', collection.count())


Chroma count before indexing: 0 / 138464
Indexing 138464 new docs (0 already present)
Indexed 500/138464 new
Indexed 1000/138464 new
Indexed 1500/138464 new
Indexed 2000/138464 new
Indexed 2500/138464 new
Indexed 3000/138464 new
Indexed 3500/138464 new
Indexed 4000/138464 new
Indexed 4500/138464 new
Indexed 5000/138464 new
Indexed 5500/138464 new
Indexed 6000/138464 new
Indexed 6500/138464 new
Indexed 7000/138464 new
Indexed 7500/138464 new
Indexed 8000/138464 new
Indexed 8500/138464 new
Indexed 9000/138464 new
Indexed 9500/138464 new
Indexed 10000/138464 new
Indexed 10500/138464 new
Indexed 11000/138464 new
Indexed 11500/138464 new
Indexed 12000/138464 new
Indexed 12500/138464 new
Indexed 13000/138464 new
Indexed 13500/138464 new
Indexed 14000/138464 new
Indexed 14500/138464 new
Indexed 15000/138464 new
Indexed 15500/138464 new
Indexed 16000/138464 new
Indexed 16500/138464 new
Indexed 17000/138464 new
Indexed 17500/138464 new
Indexed 18000/138464 new
Indexed 18500/138464 new
Indexed 1

In [ ]:
def show_retrieved_chunks(results: dict, question: str | None = None) -> None:
    """Print ranked retrieval results before LLM generation."""
    ids = results.get('ids', [[]])[0]
    documents = results.get('documents', [[]])[0]
    metadatas = results.get('metadatas', [[]])[0]
    distances = results.get('distances', [[]])[0]

    if question:
        print(f'Question: {question}\n')
    print(f'Retrieved {len(documents)} chunk(s):\n')
    print('=' * 80)

    for rank, (doc_id, text, meta) in enumerate(zip(ids, documents, metadatas), start=1):
        meta = meta or {}
        dist = distances[rank - 1] if rank - 1 < len(distances) else None
        dist_str = f'{dist:.4f}' if dist is not None else 'n/a'
        print(f'[{rank}] id={doc_id}  distance={dist_str}')
        print(
            f'    focal={meta.get("focal", "")}  slice={meta.get("slice", "")}  '
            f'owl_class={meta.get("owl_class", "")}  n_lines={meta.get("n_lines", "")}'
        )
        print('-' * 80)
        print(text)
        print('=' * 80)
        print()


def rag_answer(
    question: str,
    top_k: int = 10,
    show_retrieved: bool = True,
) -> str:
    results = collection.query(query_texts=[question], n_results=top_k)

    if show_retrieved:
        show_retrieved_chunks(results, question=question)

    context = '\n\n'.join(results['documents'][0])

    prompt = f'''
You answer questions using ONLY the facts in the context.
If the answer is not present, say you don't know.

Context blocks are verbalized KG chunks. A match may appear as several slices
(e.g. ids like `match/3895052::slice::match_core` and `match/3895052::slice::events_01`);
combine information across slices when needed. Metadata includes focal, slice,
and owl_class.

Literal ids inside each block (event/..., match/..., team/..., player/...) are
authoritative; quote them verbatim when you reference them.

Context:
{context}

Question: {question}
Answer:
'''.strip()

    if show_retrieved:
        print('--- LLM answer ---\n')

    return call_llm(prompt)


In [ ]:
# Example query
question = 'Which teams played in the match 3895052?'

print(rag_answer(question, top_k=2))

Question: Which teams played in the match 3895052?

Retrieved 2 chunk(s):

[1] id=match/3895302::slice::events_18::part::0  distance=489.6399
    focal=match/3895302  slice=events_18  owl_class=Match  n_lines=80
--------------------------------------------------------------------------------
Match match/3895302. Events (80): event/317c9477-e052-4275-bdb1-0b31ce9205d7, event/549efbe8-71ed-4a34-8813-c2f5d4551228, event/71e95621-21bf-4190-a0d9-b7dc7b869559, event/94aa3841-c664-4fd1-99b7-6f131dd6ad2c, event/46f501f7-403f-43bc-9cde-bddd974f3604, event/f72b2d61-e8bc-4f88-8815-40baf429771b, event/81f6b116-0f36-469e-9eee-7904afa9e3d2, event/9f3c2fda-cf1c-45ea-817c-00a46d4fc072, event/0bc01d0a-c95e-42a0-ae88-631bd4792d9a, event/ef835db3-9c4e-4be1-973a-010f07cbd43d, event/53865694-11bb-48af-ac9b-fd121116e215, event/fc2eaf94-f840-4088-8912-35534e0e97c8, event/ae248cbb-fdcf-408a-98dc-7085f357e811, event/a7bcdc1a-1f5e-42f6-915c-cb7ec265ba96, event/656a0def-e56e-41ed-a3a5-6e86225815a3, event/86168f3

In [ ]:
question = 'Which players are in the RB Leipzig squad?'

print(rag_answer(question, top_k=20)) 

Question: Which players are in the RB Leipzig squad?

Retrieved 20 chunk(s):

[1] id=event/ca5b4fac-1d0f-4f50-8773-ce8921925c11::part::0  distance=641.0734
    focal=event/ca5b4fac-1d0f-4f50-8773-ce8921925c11  slice=default  owl_class=Subtitution  n_lines=13
--------------------------------------------------------------------------------
Event event/ca5b4fac-1d0f-4f50-8773-ce8921925c11 is a substitution at 01:14:10.634 during the second_half in match match/3895202. Player coming on: Nicolas Seiwald (RB Leipzig). Player going off: Nicolas Seiwald. Substitution recorded at 01:14:10.634. Referenced by: match/3895202 [events]; match/3895202 [event_substitutions].

[2] id=event/c348c12c-c983-458d-9209-5b0ec9ac44d9::part::0  distance=645.4704
    focal=event/c348c12c-c983-458d-9209-5b0ec9ac44d9  slice=default  owl_class=Pass  n_lines=19
--------------------------------------------------------------------------------
Event event/c348c12c-c983-458d-9209-5b0ec9ac44d9 is a pass, performed by pla